# TEXT ENCODING NOTEBOOK
Complete demonstration of text preprocessing and encoding

In [ ]:
# Install and Import

# Install required packages (if needed)
# !pip install transformers torch numpy tqdm

import torch
import numpy as np
from transformers import (
    CLIPTextModel, CLIPTokenizer,
    BertModel, BertTokenizer
)
from tqdm import tqdm
import matplotlib.pyplot as plt

print("✅ Imports complete")

In [ ]:
# Load the Text Encoder System

# Load the complete system
%run text_encoder_system.py

print("✅ Text Encoder System loaded")


In [ ]:
# Example 1 - Basic CLIP Encoding

print("=" * 80)
print("EXAMPLE 1: Basic CLIP Text Encoding")
print("=" * 80)

# Initialize CLIP encoder
encoder = UniversalTextEncoder(
    model_name='clip',
    model_variant='openai/clip-vit-base-patch32',
    device='auto'
)

# Sample texts
texts = [
    "a beautiful sunset over the ocean",
    "a cat sitting on a windowsill",
    "a modern cityscape at night",
    "a painting of mountains and trees"
]

print(f"\n📝 Input Texts ({len(texts)}):")
for i, text in enumerate(texts, 1):
    print(f"   {i}. '{text}'")

# Encode
embeddings = encoder.encode(texts)

print(f"\n Results:")
print(f"   Input: {len(texts)} texts")
print(f"   Output shape: {embeddings.shape}")
print(f"   Embedding dimension: {encoder.get_embedding_dim()}")
print(f"   Device: {embeddings.device}")
print(f"   Data type: {embeddings.dtype}")

# Check normalization
norms = torch.norm(embeddings, dim=1)
print(f"\n   Embedding norms (should be ~1.0 if normalized):")
for i, norm in enumerate(norms):
    print(f"      Text {i+1}: {norm.item():.4f}")

In [ ]:
# Example 2 - Text Preprocessing

print("\n" + "=" * 80)
print("EXAMPLE 2: Text Preprocessing")
print("=" * 80)

# Initialize preprocessor
preprocessor = AdvancedTextPreprocessor(
    lowercase=True,
    remove_punctuation=False,
    max_length=77
)

# Test texts (with various issues)
test_texts = [
    "A   Beautiful    Sunset!!!",  # Extra spaces, caps, punctuation
    "  cat on windowsill  ",  # Leading/trailing spaces
    "Modern City @ Night #urban",  # Special characters
    "painting of MOUNTAINS & trees"  # Mixed case
]

print(f"\n🔧 Preprocessing Examples:")
for original in test_texts:
    cleaned = preprocessor.clean_text(original)
    enhanced = preprocessor.enhance_prompt(cleaned)
    
    print(f"\n   Original:  '{original}'")
    print(f"   Cleaned:   '{cleaned}'")
    print(f"   Enhanced:  '{enhanced}'")


In [ ]:
# Example 3 - Complete Pipeline

print("\n" + "=" * 80)
print("EXAMPLE 3: Complete Pipeline (Preprocess + Encode)")
print("=" * 80)

# Initialize pipeline
pipeline = TextEncoderPipeline(
    encoder_model='clip',
    preprocess=True,
    enhance_prompts=True,
    device='auto'
)

# Sample text
text = "beautiful mountain landscape"

print(f"\n Input: '{text}'")

# Process and encode
result = pipeline.process_and_encode(
    text,
    return_tokens=True,
    return_stats=True
)

print(f"\n Pipeline Results:")
print(f"   Processed text: '{result['processed_texts'][0]}'")
print(f"   Embedding shape: {result['embeddings'].shape}")
print(f"   Statistics:")
for key, value in result['stats'][0].items():
    print(f"      {key}: {value}")

print(f"\n   Token IDs (first 10): {result['tokens']['input_ids'][0][:10].tolist()}")


In [ ]:
# Example 4 - Batch Processing

print("\n" + "=" * 80)
print("EXAMPLE 4: Batch Processing")
print("=" * 80)

# Generate large batch
num_texts = 200
batch_texts = [f"sample text about art number {i}" for i in range(num_texts)]

print(f"\n Processing {num_texts} texts in batches...")

# Encode in batches
batch_embeddings = encoder.encode_batch(
    batch_texts,
    batch_size=32,
    show_progress=True
)

print(f"\n Batch Processing Complete:")
print(f"   Total texts: {len(batch_texts)}")
print(f"   Output shape: {batch_embeddings.shape}")
print(f"   Memory usage: {batch_embeddings.element_size() * batch_embeddings.nelement() / 1024**2:.2f} MB")

In [ ]:
# Example 5 - Compare Different Encoders

print("\n" + "=" * 80)
print("EXAMPLE 5: Comparing Different Encoders")
print("=" * 80)

test_text = "a beautiful painting of a sunset over mountains"

print(f"\n Test text: '{test_text}'")

# Compare CLIP vs BERT
encoders_to_test = {
    'CLIP': ('clip', 'openai/clip-vit-base-patch32'),
    'BERT': ('bert', 'bert-base-uncased')
}

results = {}

for name, (model_name, variant) in encoders_to_test.items():
    print(f"\n{name}:")
    
    enc = UniversalTextEncoder(
        model_name=model_name,
        model_variant=variant,
        device='auto'
    )
    
    emb = enc.encode([test_text])
    
    print(f"   Embedding dimension: {enc.get_embedding_dim()}")
    print(f"   Embedding norm: {emb.norm().item():.4f}")
    print(f"   First 5 values: {emb[0][:5].tolist()}")
    
    results[name] = emb

# Compute similarity between encoders
if len(results) >= 2:
    embeddings_list = list(results.values())
    similarity = torch.nn.functional.cosine_similarity(
        embeddings_list[0], embeddings_list[1], dim=1
    )
    print(f"\n Cosine similarity between encoders: {similarity.item():.4f}")

In [ ]:
# Example 6 - Analyze Dataset Captions

print("\n" + "=" * 80)
print("EXAMPLE 6: Analyzing Dataset Captions")
print("=" * 80)

# Sample captions (you can replace with real dataset)
sample_captions = [
    "a beautiful landscape with mountains",
    "a cat sitting on a chair",
    "a modern building in the city",
    "a painting of flowers in a vase",
    "a person walking in the park",
    "an abstract artwork with colors",
    "a sunset over the ocean",
    "a detailed drawing of a face"
] * 25  # Repeat to get 200 samples

print(f"\n Analyzing {len(sample_captions)} captions...")

# Analyze
preprocessor = AdvancedTextPreprocessor()
stats = []

for caption in tqdm(sample_captions, desc="Analyzing"):
    stat = preprocessor.analyze_text(caption)
    stats.append(stat)

# Aggregate
lengths = [s['length'] for s in stats]
word_counts = [s['word_count'] for s in stats]

print(f"\n Statistics:")
print(f"   Character Length:")
print(f"      Min: {min(lengths)}, Max: {max(lengths)}")
print(f"      Mean: {np.mean(lengths):.1f}, Median: {np.median(lengths):.1f}")
print(f"   ")
print(f"   Word Count:")
print(f"      Min: {min(word_counts)}, Max: {max(word_counts)}")
print(f"      Mean: {np.mean(word_counts):.1f}, Median: {np.median(word_counts):.1f}")

In [ ]:
# Example 7 - Visualize Embeddings

print("\n" + "=" * 80)
print("EXAMPLE 7: Visualize Embeddings")
print("=" * 80)

# Encode several texts
vis_texts = [
    "a cat",
    "a kitten",
    "a dog",
    "a puppy",
    "a sunset",
    "a sunrise",
    "a mountain",
    "a hill"
]

print(f"\n Encoding {len(vis_texts)} texts for visualization...")

vis_embeddings = encoder.encode(vis_texts)

# Compute similarity matrix
similarity_matrix = torch.matmul(vis_embeddings, vis_embeddings.T)

# Plot
plt.figure(figsize=(10, 8))
plt.imshow(similarity_matrix.cpu().numpy(), cmap='viridis', aspect='auto')
plt.colorbar(label='Cosine Similarity')
plt.xticks(range(len(vis_texts)), vis_texts, rotation=45, ha='right')
plt.yticks(range(len(vis_texts)), vis_texts)
plt.title('Text Embedding Similarity Matrix')
plt.tight_layout()
plt.savefig('embedding_similarity.png', dpi=150, bbox_inches='tight')
print(" Saved: embedding_similarity.png")
plt.show()

print(f"\n Observations:")
print(f"   • Similar words (cat/kitten, dog/puppy) have higher similarity")
print(f"   • Related concepts (sunset/sunrise) are also similar")
print(f"   • Unrelated concepts (cat/sunset) have lower similarity")

In [ ]:
# Example 8 - Embedding Cache

print("\n" + "=" * 80)
print("EXAMPLE 8: Using Embedding Cache")
print("=" * 80)

# Test caching
cache_texts = ["test text 1", "test text 2", "test text 3"]

print("\n  First encoding (no cache):")
import time
start = time.time()
emb1 = encoder.encode(cache_texts, use_cache=True)
time1 = time.time() - start
print(f"   Time: {time1:.4f}s")

print("\n  Second encoding (with cache):")
start = time.time()
emb2 = encoder.encode(cache_texts, use_cache=True)
time2 = time.time() - start
print(f"   Time: {time2:.4f}s")

print(f"\n Speedup: {time1/time2:.2f}x faster with cache")
print(f"   Embeddings identical: {torch.allclose(emb1, emb2)}")

# Save cache
encoder.save_cache('./embedding_cache.json')


In [ ]:
# Example 9 - Integration with Text-to-Image Model

print("\n" + "=" * 80)
print("EXAMPLE 9: Integration with Text-to-Image Model")
print("=" * 80)

# This shows how to use the encoder with your GAN

print("""
 Integration Example:

# Initialize encoder
text_encoder = UniversalTextEncoder(
    model_name='clip',
    model_variant='openai/clip-vit-base-patch32'
)

# In your training loop:
for batch in dataloader:
    images = batch['images']
    texts = batch['captions']
    
    # Encode texts
    text_embeddings = text_encoder.encode(texts, normalize=True)
    
    # Use embeddings with your GAN
    noise = torch.randn(batch_size, latent_dim)
    generated_images = generator(noise, text_embeddings)
    
    # Train discriminator
    real_score = discriminator(images, text_embeddings)
    fake_score = discriminator(generated_images, text_embeddings)
    
    # ... rest of training code

 The encoder handles:
   - Text cleaning and preprocessing
   - Tokenization
   - Embedding generation
   - Normalization
   - Batch processing
   - Caching for efficiency
""")

In [ ]:
# SUMMARY

print("\n" + "=" * 80)
print(" TEXT ENCODING SYSTEM - COMPLETE!")
print("=" * 80)

print("""
 What You Learned:

1. ✓ Basic text encoding with CLIP
2. ✓ Text preprocessing and cleaning
3. ✓ Complete pipeline (preprocess + encode)
4. ✓ Batch processing for efficiency
5. ✓ Comparing different encoders
6. ✓ Analyzing dataset captions
7. ✓ Visualizing embeddings
8. ✓ Using embedding cache
9. ✓ Integration with text-to-image models

 Key Features:

• Multiple encoder support (CLIP, BERT, T5, GPT-2)
• Advanced preprocessing
• Automatic normalization
• Batch processing
• Embedding caching
• Statistical analysis
• Easy integration

 Next Steps:

1. Use this system in your text-to-image GAN training
2. Try different encoder models
3. Experiment with preprocessing options
4. Analyze your specific dataset captions
5. Optimize for your use case

 Files Generated:
   • embedding_similarity.png
   • embedding_cache.json
""")